# 03 · Gold (local + pandas) — pronta para negócio e preparada para IA

```
SILVER ──► GOLD (Parquet no lake, fonte da verdade)
              ├── Parte A · tabelas de negócio ──► Postgres (schema gold) ──► Metabase
              └── Parte B · tabela de chunks   ──► [próximo módulo: embeddings + banco vetorial]
```

A Gold responde a perguntas de negócio, tem **dono, grão e contrato**, e só é publicada se passar nos **gates de qualidade**.

**O que NÃO faremos aqui:** criar o banco vetorial. Na Parte B deixamos a arquitetura **pronta para recebê-lo**: tabela de chunks, PII mascarada, `content_hash` para reprocessar só o que mudou e o contrato da tabela `pgvector`. O banco em si é construído no próximo módulo.

**Antes de começar:** rode os notebooks 01 e 02 (versões pandas), com o MinIO no ar. A parte "Servir ao Postgres" só funciona na aula, quando o Postgres estiver no ar.

## Passo 0 · Conexão, funções de apoio e mapa de colunas

In [1]:
import hashlib
import io
import json
import os
import re
from datetime import datetime, timezone

import boto3
import numpy as np
import pandas as pd

ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://localhost:9000")
USUARIO = os.environ.get("MINIO_USER", "admin")
SENHA = os.environ.get("MINIO_PASSWORD", "admin12345")

s3 = boto3.client("s3", endpoint_url=ENDPOINT, aws_access_key_id=USUARIO,
                  aws_secret_access_key=SENHA, region_name="us-east-1")


def listar(bucket, prefixo=""):
    achados = []
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefixo):
        achados += [(o["Key"], o["Size"]) for o in pagina.get("Contents", [])]
    return achados


def ler_prefixo(bucket, prefixo):
    partes = [pd.read_parquet(io.BytesIO(s3.get_object(Bucket=bucket, Key=c)["Body"].read()))
              for c, _ in listar(bucket, prefixo)]
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


def gravar_parquet(df, chave):
    buf = io.BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow")
    s3.put_object(Bucket="warehouse", Key=chave, Body=buf.getvalue())


def gravar_gold(df, tabela):
    for chave, _ in listar("warehouse", f"gold/{tabela}/"):
        s3.delete_object(Bucket="warehouse", Key=chave)          # Gold é reconstruída por inteiro
    gravar_parquet(df, f"gold/{tabela}/part-0000.parquet")
    return len(df)


def para_int(serie):
    return pd.to_numeric(serie, errors="coerce").astype("Int64")


# Mapa de colunas: se o nome na SUA Silver for outro, ajuste aqui e só aqui.
AV_PRODUTO = "produto_id"      # avaliacoes: produto avaliado
AV_TEXTO = "comentario"        # avaliacoes: texto livre
VALIDOS = ["pago", "enviado", "entregue"]          # status que contam como venda
CANCELADOS = ["cancelado", "devolvido"]


def exigir(df, colunas, nome):
    faltam = [c for c in colunas if c not in df.columns]
    if faltam:
        raise KeyError(f"{nome}: faltam as colunas {faltam}. Colunas existentes: {list(df.columns)}. Ajuste o mapa de colunas.")


print("conectado em", ENDPOINT)

conectado em http://localhost:9000


## Parte A · Gold para o negócio

### Passo 1 · O contrato vem antes do código
Antes de calcular qualquer coisa, definimos **o que cada tabela é**. Sem grão e sem dono, o mesmo número aparece diferente em dois painéis.

| Tabela | Grão (1 linha =) | Regra de negócio | Dado pessoal |
|---|---|---|---|
| `vendas_diarias` | um dia | receita = soma de `valor_item` dos pedidos `pago`, `enviado` e `entregue`; cancelados e devolvidos são contados à parte | não |
| `clientes_360` | um cliente | métricas de compra; **nome mascarado**, sem e-mail | sim, minimizado |
| `avaliacoes_resumo` | um produto | média das notas e % de notas baixas (≤ 2) | não |

O contrato fica **no código**, não só no texto, porque os gates e o dicionário de dados vão usá-lo.

In [2]:
CONTRATOS = {
    "vendas_diarias": {
        "grao": "um dia", "dono": "Comercial (exemplo: preencha com o responsável)",
        "atualizacao": "reconstruída a cada execução", "pii": False,
        "descricao": "Vendas por dia. Receita considera só pedidos pago/enviado/entregue.",
        "colunas": {
            "data": "Dia do pedido", "pedidos": "Pedidos válidos no dia", "itens": "Itens vendidos no dia",
            "receita": "Soma de valor_item dos pedidos válidos (R$)", "ticket_medio": "receita / pedidos (R$)",
            "pedidos_cancelados": "Pedidos cancelados ou devolvidos no dia", "_gold_ts": "Quando a Gold foi gerada"},
    },
    "clientes_360": {
        "grao": "um cliente", "dono": "CRM (exemplo: preencha com o responsável)",
        "atualizacao": "reconstruída a cada execução", "pii": True,
        "descricao": "Visão do cliente. O nome é mascarado e o e-mail não entra na Gold (minimização).",
        "colunas": {
            "cliente_id": "Identificador do cliente", "nome_mascarado": "Nome com letras ocultas",
            "estado": "UF do cadastro", "num_pedidos": "Pedidos válidos do cliente", "valor_total": "Receita do cliente (R$)",
            "ticket_medio": "valor_total / num_pedidos (R$)", "primeira_compra": "Data da primeira compra",
            "ultima_compra": "Data da última compra", "dias_desde_ultima_compra": "Dias entre a última compra e o último dia da base",
            "segmento": "sem_compra, ativo (até 90 dias) ou inativo", "_gold_ts": "Quando a Gold foi gerada"},
    },
    "avaliacoes_resumo": {
        "grao": "um produto", "dono": "Produto (exemplo: preencha com o responsável)",
        "atualizacao": "reconstruída a cada execução", "pii": False,
        "descricao": "Satisfação por produto.",
        "colunas": {
            "produto_id": "Identificador do produto", "produto_nome": "Nome do produto",
            "qtd_avaliacoes": "Número de avaliações", "nota_media": "Média das notas (1 a 5)",
            "pct_notas_baixas": "% de avaliações com nota 1 ou 2", "_gold_ts": "Quando a Gold foi gerada"},
    },
}
pd.DataFrame([{"tabela": t, "grao": c["grao"], "dono": c["dono"], "pii": c["pii"]} for t, c in CONTRATOS.items()])

,tabela,grao,dono,pii
0,vendas_diarias,um dia,Comercial (exemplo: preencha com o responsável),False
1,clientes_360,um cliente,CRM (exemplo: preencha com o responsável),True
2,avaliacoes_resumo,um produto,Produto (exemplo: preencha com o responsável),False


### Passo 2 · Ler a Silver

In [3]:
cli = ler_prefixo("warehouse", "silver/clientes/")
prod = ler_prefixo("warehouse", "silver/produtos/")
ped = ler_prefixo("warehouse", "silver/pedidos/")
itens = ler_prefixo("warehouse", "silver/itens_pedido/")
av = ler_prefixo("warehouse", "silver/avaliacoes/")

exigir(ped, ["pedido_id", "cliente_id", "data_pedido", "status"], "silver.pedidos")
exigir(itens, ["pedido_id", "valor_item", "status", "data_pedido"], "silver.itens_pedido")
exigir(av, ["avaliacao_id", "nota", AV_PRODUTO, AV_TEXTO], "silver.avaliacoes")
exigir(cli, ["cliente_id", "nome", "estado"], "silver.clientes")

for nome, df in (("clientes", cli), ("produtos", prod), ("pedidos", ped), ("itens_pedido", itens), ("avaliacoes", av)):
    print(f"{nome:<13}{len(df):>8,} linhas")

clientes        5,000 linhas
produtos          197 linhas
pedidos        29,960 linhas
itens_pedido   49,283 linhas
avaliacoes      6,464 linhas


### Passo 3 · Construir as tabelas
Duas decisões de governança já aparecem no código:
- **Nome mascarado:** a Gold é lida por muita gente (Metabase). Quem só precisa de "quanto o cliente comprou" não precisa do nome inteiro.
- **E-mail fica de fora:** o que não é necessário para a pergunta de negócio não entra na Gold.

In [4]:
def mascarar_nome(nome):
    """'Ana Souza' -> 'A** S****' (mantém a inicial de cada palavra)."""
    if not isinstance(nome, str):
        return None
    return " ".join(p[0] + "*" * (len(p) - 1) for p in nome.split())


AGORA = datetime.now(timezone.utc)

itens = itens.copy()
itens["data"] = pd.to_datetime(itens["data_pedido"]).dt.date
validos = itens[itens["status"].isin(VALIDOS)]

# vendas_diarias
vendas = (validos.groupby("data")
          .agg(pedidos=("pedido_id", "nunique"), itens=("pedido_id", "size"), receita=("valor_item", "sum"))
          .reset_index())
vendas["receita"] = vendas["receita"].round(2)
vendas["ticket_medio"] = (vendas["receita"] / vendas["pedidos"]).round(2)
canc = ped[ped["status"].isin(CANCELADOS)].assign(data=lambda d: pd.to_datetime(d["data_pedido"]).dt.date)
canc = canc.groupby("data").size().rename("pedidos_cancelados").reset_index()
vendas = vendas.merge(canc, on="data", how="outer").fillna(0)
for c in ("pedidos", "itens", "pedidos_cancelados"):
    vendas[c] = vendas[c].astype(int)
vendas = vendas.sort_values("data").reset_index(drop=True)

# clientes_360
por_cli = (validos.groupby("cliente_id")
           .agg(num_pedidos=("pedido_id", "nunique"), valor_total=("valor_item", "sum"),
                primeira_compra=("data", "min"), ultima_compra=("data", "max"))
           .reset_index())
por_cli["cliente_id"] = para_int(por_cli["cliente_id"])
c360 = cli[["cliente_id", "nome", "estado"]].merge(por_cli, on="cliente_id", how="left")
c360["nome_mascarado"] = c360["nome"].map(mascarar_nome)
c360 = c360.drop(columns="nome")                                   # o nome completo não sai da Silver
c360["num_pedidos"] = c360["num_pedidos"].fillna(0).astype(int)
c360["valor_total"] = c360["valor_total"].fillna(0).round(2)
c360["ticket_medio"] = (c360["valor_total"] / c360["num_pedidos"].where(c360["num_pedidos"] > 0)).round(2)
ref = pd.to_datetime(itens["data"]).max()                          # "hoje" = último dia da base (resultado reprodutível)
c360["dias_desde_ultima_compra"] = (ref - pd.to_datetime(c360["ultima_compra"])).dt.days
c360["segmento"] = np.select([c360["num_pedidos"] == 0, c360["dias_desde_ultima_compra"] <= 90],
                             ["sem_compra", "ativo"], default="inativo")
c360 = c360[["cliente_id", "nome_mascarado", "estado", "num_pedidos", "valor_total", "ticket_medio",
             "primeira_compra", "ultima_compra", "dias_desde_ultima_compra", "segmento"]]

# avaliacoes_resumo
av = av.copy()
av[AV_PRODUTO] = para_int(av[AV_PRODUTO])
resumo_av = (av.groupby(AV_PRODUTO)
             .agg(qtd_avaliacoes=("avaliacao_id", "size"), nota_media=("nota", "mean"),
                  pct_notas_baixas=("nota", lambda s: 100 * (s <= 2).mean()))
             .reset_index().rename(columns={AV_PRODUTO: "produto_id"}))
resumo_av["nota_media"] = resumo_av["nota_media"].round(2)
resumo_av["pct_notas_baixas"] = resumo_av["pct_notas_baixas"].round(1)
nomes = prod[["produto_id", "nome"]].rename(columns={"nome": "produto_nome"})
resumo_av = resumo_av.merge(nomes, on="produto_id", how="left")[
    ["produto_id", "produto_nome", "qtd_avaliacoes", "nota_media", "pct_notas_baixas"]]

gold = {"vendas_diarias": vendas, "clientes_360": c360, "avaliacoes_resumo": resumo_av}
for t, df in gold.items():
    df["_gold_ts"] = AGORA
    print(f"{t:<18}{len(df):>7,} linhas")
c360.head(5)

vendas_diarias        541 linhas
clientes_360        5,000 linhas
avaliacoes_resumo     200 linhas


,cliente_id,nome_mascarado,estado,num_pedidos,valor_total,ticket_medio,primeira_compra,ultima_compra,dias_desde_ultima_compra,segmento,_gold_ts
0,1,A**** P******,BA,11,10104.20,918.56,2025-03-29,2025-11-12,230.0,inativo,2026-10-26 16:22:02.878821+00:00
1,3337,B****** R******,BA,17,26393.36,1552.55,2025-03-03,2026-05-30,31.0,ativo,2026-10-26 16:22:02.878821+00:00
2,3336,M***** A****,BA,2,3165.70,1582.85,2026-06-08,2026-06-10,20.0,ativo,2026-10-26 16:22:02.878821+00:00
3,3335,A**** P******,PE,6,9435.94,1572.66,2025-11-28,2026-05-24,37.0,ativo,2026-10-26 16:22:02.878821+00:00
4,3334,J*** S*****,PE,5,3160.10,632.02,2025-04-30,2026-05-30,31.0,ativo,2026-10-26 16:22:02.878821+00:00


### Passo 4 · Gates: a Gold só é publicada se passar
Um gate é uma verificação que **bloqueia** a publicação. Três tipos:
1. **Grão e chaves:** a tabela respeita o contrato?
2. **Reconciliação:** o total da Gold bate com a Silver, calculado de outro jeito?
3. **Governança:** toda coluna tem descrição? Algum dado pessoal que não devia estar aqui?

In [7]:
PII_PROIBIDA = {"nome", "email", "cpf", "telefone"}      # colunas que nunca podem aparecer na Gold


def checar(tabela, df):
    c = CONTRATOS[tabela]
    regras = [("todas as colunas têm descrição no contrato", set(df.columns) <= set(c["colunas"])),
              ("nenhuma coluna de PII bruta", not (PII_PROIBIDA & set(df.columns)))]
    if tabela == "vendas_diarias":
        regras += [("grão: 1 linha por data", not df["data"].duplicated().any()),
                   ("data sem nulos", df["data"].notna().all()), ("receita >= 0", (df["receita"] >= 0).all())]
    if tabela == "clientes_360":
        regras += [("grão: 1 linha por cliente", not df["cliente_id"].duplicated().any()),
                   ("cliente_id sem nulos", df["cliente_id"].notna().all()),
                   ("todos os clientes da Silver estão aqui", len(df) == len(cli))]
    if tabela == "avaliacoes_resumo":
        regras += [("grão: 1 linha por produto", not df["produto_id"].duplicated().any()),
                   ("nota_media entre 1 e 5", df["nota_media"].between(1, 5).all())]
    return pd.DataFrame([{"tabela": tabela, "regra": r, "status": "OK" if bool(ok) else "ERRO"} for r, ok in regras])


def reconciliar():
    """Recalcula a receita por OUTRO caminho (pelos pedidos válidos) e compara com a Gold."""
    ids_validos = set(ped.loc[ped["status"].isin(VALIDOS), "pedido_id"]) & set(itens["pedido_id"])
    esperado = round(float(itens.loc[itens["pedido_id"].isin(ids_validos), "valor_item"].sum()), 2)
    return pd.DataFrame([
        {"tabela": "vendas_diarias", "regra": f"receita Gold = receita Silver ({esperado:,.2f})",
         "status": "OK" if abs(round(float(vendas["receita"].sum()), 2) - esperado) < 0.01 else "ERRO"},
        {"tabela": "vendas_diarias", "regra": f"pedidos Gold = pedidos válidos Silver ({len(ids_validos)})",
         "status": "OK" if int(vendas["pedidos"].sum()) == len(ids_validos) else "ERRO"}])


def rodar_gates(tabelas):
    rel = pd.concat([checar(t, df) for t, df in tabelas.items()] + [reconciliar()], ignore_index=True)
    return rel


gates = rodar_gates(gold)
print(gates["status"].value_counts().to_string())
gates

status
OK    16


,tabela,regra,status
0,vendas_diarias,todas as colunas têm descrição no contrato,OK
1,vendas_diarias,nenhuma coluna de PII bruta,OK
2,vendas_diarias,grão: 1 linha por data,OK
3,vendas_diarias,data sem nulos,OK
4,vendas_diarias,receita >= 0,OK
5,clientes_360,todas as colunas têm descrição no contrato,OK
6,clientes_360,nenhuma coluna de PII bruta,OK
7,clientes_360,grão: 1 linha por cliente,OK
8,clientes_360,cliente_id sem nulos,OK
9,clientes_360,todos os clientes da Silver estão aqui,OK


**Demonstração: o gate funcionando.** Vamos estragar uma cópia (uma linha repetida em `vendas_diarias`) e ver o que acontece.

In [8]:
ruim = pd.concat([vendas, vendas.head(1)], ignore_index=True)
r = checar("vendas_diarias", ruim)
print(r[r["status"] == "ERRO"].to_string(index=False))
print("\n-> publicação BLOQUEADA" if (r["status"] == "ERRO").any() else "-> publicação liberada")

        tabela                  regra status
vendas_diarias grão: 1 linha por data   ERRO

-> publicação BLOQUEADA


### Passo 5 · Publicar no lake
A Gold é gravada no storage (**fonte da verdade**). O contrato vai junto, como um objeto em `gold/_contratos/`, para que quem consome saiba o que está lendo.

In [9]:
if (gates["status"] == "ERRO").any():
    raise RuntimeError("Gates falharam: a Gold NÃO foi publicada.\n" + gates[gates["status"] == "ERRO"].to_string(index=False))

for t, df in gold.items():
    print(f"{t:<18}{gravar_gold(df, t):>7,} linhas -> s3://warehouse/gold/{t}/")

s3.put_object(Bucket="warehouse", Key="gold/_contratos/contratos.json",
              Body=json.dumps(CONTRATOS, ensure_ascii=False, indent=2).encode("utf-8"))
print("\ncontrato publicado em s3://warehouse/gold/_contratos/contratos.json")
for chave, tam in listar("warehouse", "gold/"):
    print(f"{tam:>9,}  {chave}")

vendas_diarias        541 linhas -> s3://warehouse/gold/vendas_diarias/
clientes_360        5,000 linhas -> s3://warehouse/gold/clientes_360/
avaliacoes_resumo     200 linhas -> s3://warehouse/gold/avaliacoes_resumo/

contrato publicado em s3://warehouse/gold/_contratos/contratos.json
    2,096  gold/_contratos/contratos.json
    7,942  gold/avaliacoes_resumo/part-0000.parquet
  133,407  gold/clientes_360/part-0000.parquet
   17,260  gold/vendas_diarias/part-0000.parquet


### Passo 6 · Servir ao Postgres e ao Metabase *(na aula)*
O Metabase não lê Parquet: ele fala com um banco SQL. Por isso **copiamos** a Gold para o Postgres (schema `gold`).
O lake continua sendo a fonte da verdade; o Postgres é uma **cópia de serviço** que pode ser apagada e refeita.

Duas lições de governança:
- O Metabase deve usar um usuário **somente leitura**. Quem faz BI não precisa (nem deve) poder alterar a Gold.
- `if_exists="replace"` apaga a tabela e **perde as permissões**. Por isso concedemos `SELECT` por padrão no schema.

*Se o Postgres ainda não estiver no ar, esta célula avisa e você segue para a Parte B.*

In [11]:
from sqlalchemy import create_engine, text

PG_HOST = os.environ.get("POSTGRES_HOST", "localhost")
PG_PORT = os.environ.get("POSTGRES_PORT", "5432")
PG_DB = os.environ.get("POSTGRES_DB", "lake")                 # ajuste ao docker-compose da aula
PG_USER = os.environ.get("POSTGRES_USER", "aula")
PG_SENHA = os.environ.get("POSTGRES_PASSWORD", "aula123")     # só laboratório; em produção, cofre de segredos
RO_USER = "metabase_ro"
RO_SENHA = os.environ.get("METABASE_PG_PASSWORD", "leitor123")


def publicar_postgres(tabelas, schema="gold"):
    eng = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_SENHA}@{PG_HOST}:{PG_PORT}/{PG_DB}")
    with eng.begin() as con:
        con.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema}"))
    for nome, df in tabelas.items():
        df.to_sql(nome, eng, schema=schema, if_exists="replace", index=False, method="multi", chunksize=1000)
    with eng.begin() as con:
        con.execute(text(f"""
            DO $$ BEGIN
              IF NOT EXISTS (SELECT FROM pg_roles WHERE rolname = '{RO_USER}') THEN
                CREATE ROLE {RO_USER} LOGIN PASSWORD '{RO_SENHA}';
              END IF;
            END $$;"""))
        con.execute(text(f"GRANT USAGE ON SCHEMA {schema} TO {RO_USER}"))
        con.execute(text(f"GRANT SELECT ON ALL TABLES IN SCHEMA {schema} TO {RO_USER}"))
        con.execute(text(f"ALTER DEFAULT PRIVILEGES IN SCHEMA {schema} GRANT SELECT ON TABLES TO {RO_USER}"))
    return eng


try:
    eng = publicar_postgres(gold)
    with eng.connect() as con:
        for t in gold:
            n = con.execute(text(f"SELECT count(*) FROM gold.{t}")).scalar()
            print(f"gold.{t:<18}{n:>7,} linhas no Postgres")
except Exception as e:
    print("Postgres ainda não está disponível — pule esta célula por enquanto.")
    print("Motivo:", str(e).splitlines()[0][:160])

Postgres ainda não está disponível — pule esta célula por enquanto.
Motivo: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: FATAL:  database "lake" does not exist


**Conectar o Metabase** (<http://localhost:3000>): *Admin → Databases → Add database → PostgreSQL*
- **Host:** `postgres` (nome do serviço no Docker; use `localhost` se o Metabase não estiver em container)
- **Port:** `5432` · **Database:** o do compose · **Username:** `metabase_ro` · **Password:** a de `METABASE_PG_PASSWORD`
- Depois de salvar, use *Sync database schema now*. Crie uma pergunta: receita por dia a partir de `gold.vendas_diarias`.

**Teste de governança:** tente criar uma tabela ou apagar uma linha usando o usuário `metabase_ro`. Deve falhar. Por quê é bom que falhe?

## Parte B · Gold preparada para IA

Um banco vetorial guarda **trechos de texto (chunks)** e seus **embeddings**, para busca por significado. O banco só é criado no próximo módulo, mas as decisões que o tornam seguro e barato **precisam ser tomadas agora**, na Gold:

```
 Silver ──► [mascarar PII] ──► [montar chunks] ──► [content_hash] ──► gold/chunks (Parquet no lake)
                                                                              │
                                              próximo módulo ─────────────────┘
                                              embeddings ──► banco vetorial (pgvector)
```

| Decisão | Por que agora |
|---|---|
| **PII mascarada antes** | um embedding não se "desfaz": se o telefone entrou no vetor, ele está lá. Depois de gerado, o jeito é refazer. |
| **`content_hash`** | gerar embedding custa dinheiro. Com o hash, reprocessamos **só o que mudou**. |
| **Lake como fonte da verdade** | o banco vetorial é uma cópia derivada: dá para recriá-lo a partir dos chunks. |
| **Contrato da tabela `pgvector`** | o próximo módulo recebe um formato combinado, não uma surpresa. |

### Passo 7 · Classificar o dado pessoal e definir a política
Antes de mascarar, decidimos **o que é dado pessoal** e **o que fazer com ele**.

| Campo | Classe | Política |
|---|---|---|
| `nome` | identificador direto | mascarado na Gold; **nunca** entra em chunk |
| `email`, CPF, telefone | identificador direto | fora da Gold; se aparecer em texto livre, vira `[EMAIL]`, `[CPF]`, `[TELEFONE]` |
| `cliente_id` | pseudônimo (permite ligar à pessoa) | **fora** dos metadados do chunk |
| `comentario` (texto livre) | pode conter qualquer coisa | varrido e mascarado |
| `estado`, `produto_id`, `nota` | não identificam sozinhos | podem ir como metadado |

**Limite honesto:** expressões regulares pegam e-mail, CPF e telefone, mas **não pegam nomes próprios** escritos no meio de uma frase. Isso exige reconhecimento de entidades (NER), que fica para o módulo de IA. Por isso existe o gate de varredura no Passo 10.

In [ ]:
PADROES_PII = [
    ("EMAIL", re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")),
    ("CPF", re.compile(r"(?<!\d)\d{3}\.?\d{3}\.?\d{3}-?\d{2}(?!\d)")),          # antes do telefone, que é mais genérico
    ("TELEFONE", re.compile(r"(?<!\d)(?:\+?55\s?)?(?:\(?\d{2}\)?\s?)?9?\d{4}[-\s]?\d{4}(?!\d)")),
]
VERSAO_POLITICA_PII = "v1"


def mascarar_pii(texto):
    if not isinstance(texto, str):
        return ""
    for rotulo, padrao in PADROES_PII:
        texto = padrao.sub(f"[{rotulo}]", texto)
    return texto


def tem_pii(texto):
    return any(p.search(texto) for _, p in PADROES_PII)


exemplos = ["Ótimo produto! Me ligue em (81) 98888-7777 ou escreva para maria.silva@exemplo.com.",
            "Meu CPF 123.456.789-09 está no cadastro, chegou rápido.",
            "Chegou em 3 dias, nota 10."]
for t in exemplos:
    print("antes :", t)
    print("depois:", mascarar_pii(t), "\n")

### Passo 8 · Montar os chunks
Aqui, **1 avaliação = 1 chunk**: o texto é curto e cabe inteiro. Textos longos (manuais, contratos) precisam ser **quebrados** em pedaços com sobreposição; essa estratégia é decidida no módulo vetorial.

Cada chunk leva o **contexto** junto (produto e nota), porque um comentário como "chegou quebrado" só faz sentido sabendo de qual produto se fala.

In [ ]:
def hash_conteudo(texto):
    normalizado = " ".join(texto.lower().split())            # ignora maiúsculas e espaços repetidos
    return hashlib.sha256(normalizado.encode("utf-8")).hexdigest()


def montar_chunks(av, prod):
    nomes = prod[["produto_id", "nome"]].rename(columns={"nome": "produto_nome"})
    df = av.assign(produto_id=para_int(av[AV_PRODUTO])).merge(nomes, on="produto_id", how="left")
    df["comentario_limpo"] = df[AV_TEXTO].map(mascarar_pii).str.strip()
    df = df[df["comentario_limpo"] != ""].copy()
    df["texto"] = ("Produto: " + df["produto_nome"].fillna("desconhecido") + " | Nota: " + df["nota"].astype(str)
                   + "/5 | Avaliação: " + df["comentario_limpo"])
    # minimização: nenhum cliente_id, nome ou e-mail nos metadados
    df["metadata"] = [json.dumps({"produto_id": None if pd.isna(p) else int(p), "nota": int(n), "origem": "silver.avaliacoes"})
                      for p, n in zip(df["produto_id"], df["nota"])]
    out = pd.DataFrame({
        "chunk_id": "avaliacao:" + df["avaliacao_id"].astype(str),
        "fonte": "silver.avaliacoes", "fonte_id": df["avaliacao_id"].astype(str),
        "texto": df["texto"].values, "metadata": df["metadata"].values,
        "content_hash": [hash_conteudo(t) for t in df["texto"]],
        "politica_pii": VERSAO_POLITICA_PII})
    return out.reset_index(drop=True)


novos = montar_chunks(av, prod)
print(len(novos), "chunks de", len(av), "avaliações (sem comentário não vira chunk)")
novos[["chunk_id", "texto", "content_hash"]].head(4)

### Passo 9 · `content_hash`: reprocessar só o que mudou
O próximo módulo vai gerar um embedding por chunk, e cada embedding custa tempo e dinheiro. Se amanhã só 3 comentários mudarem, não faz sentido refazer os outros 10.000.

A regra: comparar o `content_hash` de hoje com o da última execução.

| Situação | Ação no próximo módulo |
|---|---|
| chunk novo | gerar embedding |
| hash diferente | gerar de novo (o texto mudou) |
| hash igual | **manter** o embedding que já existe |
| chunk sumiu | remover do banco vetorial |

In [ ]:
COLS_ESTADO = ["chunk_id", "content_hash", "embedding_status", "modelo_embedding", "embedding_em"]


def sincronizar(novos, anteriores):
    """Compara com a execução anterior. Só 'novo' e 'alterado' ficam pendentes de embedding."""
    out = novos.copy()
    if anteriores.empty:
        out["acao"] = "novo"
        out["embedding_status"], out["modelo_embedding"], out["embedding_em"] = "pendente", None, pd.NaT
        removidos = anteriores
    else:
        ant = anteriores[COLS_ESTADO].rename(columns={c: c + "_ant" for c in COLS_ESTADO if c != "chunk_id"})
        out = out.merge(ant, on="chunk_id", how="left")
        out["acao"] = np.select([out["content_hash_ant"].isna(), out["content_hash"] != out["content_hash_ant"]],
                                ["novo", "alterado"], default="inalterado")
        mantem = out["acao"] == "inalterado"
        out["embedding_status"] = np.where(mantem, out["embedding_status_ant"], "pendente")
        out["modelo_embedding"] = np.where(mantem, out["modelo_embedding_ant"], None)
        out["embedding_em"] = out["embedding_em_ant"].where(mantem, pd.NaT)
        out = out.drop(columns=[c for c in out.columns if c.endswith("_ant")])
        removidos = anteriores[~anteriores["chunk_id"].isin(out["chunk_id"])]
    out["atualizado_em"] = datetime.now(timezone.utc)
    return out, removidos


# simulação (NÃO grava nada): imagine que o próximo módulo já gerou todos os embeddings...
anteriores = novos.assign(embedding_status="embedado", modelo_embedding="modelo-exemplo", embedding_em=pd.Timestamp.now(tz="UTC"))

# ...e que, no dia seguinte, um comentário mudou, um sumiu e um chegou
id_removido, id_alterado = novos["fonte_id"].iloc[0], novos["fonte_id"].iloc[1]
av_amanha = av[av["avaliacao_id"].astype(str) != id_removido].copy()
av_amanha.loc[av_amanha["avaliacao_id"].astype(str) == id_alterado, AV_TEXTO] = "Mudei de ideia: produto ruim, chegou com defeito."
av_amanha = pd.concat([av_amanha, av.head(1).assign(avaliacao_id=900001, **{AV_TEXTO: "Comentário novo de amanhã."})],
                      ignore_index=True)

amanha, removidos = sincronizar(montar_chunks(av_amanha, prod), anteriores)
print(amanha["acao"].value_counts().to_string())
print("removidos:", len(removidos))
print(f"-> embeddings a gerar: {(amanha['acao'] != 'inalterado').sum()} de {len(amanha)}")

### Passo 10 · Gates de prontidão e publicação dos chunks
Os chunks só seguem para o próximo módulo se passarem em verificações próprias. O gate de varredura procura PII **de novo**, sem confiar no mascaramento anterior.

In [ ]:
def gates_chunks(ch):
    chaves_proibidas = {"cliente_id", "nome", "email", "cpf", "telefone"}
    regras = [
        ("chunk_id único", not ch["chunk_id"].duplicated().any()),
        ("texto nunca vazio", ch["texto"].str.strip().ne("").all()),
        ("sem PII residual no texto (nova varredura)", not ch["texto"].map(tem_pii).any()),
        ("tamanho <= 1000 caracteres", (ch["texto"].str.len() <= 1000).all()),
        ("content_hash confere com o texto", (ch["texto"].map(hash_conteudo) == ch["content_hash"]).all()),
        ("metadados sem cliente_id/nome/e-mail", not any(chaves_proibidas & set(json.loads(m)) for m in ch["metadata"])),
    ]
    return pd.DataFrame([{"regra": r, "status": "OK" if bool(ok) else "ERRO"} for r, ok in regras])


# anterior = o que já foi publicado (se houver), para herdar o estado dos embeddings
anterior_pub = ler_prefixo("warehouse", "gold/chunks/")
chunks, removidos = sincronizar(novos, anterior_pub)
g = gates_chunks(chunks)
print(g.to_string(index=False))

# prova de que o gate funciona: um texto com telefone escapando do mascaramento
vazado = chunks.head(1).assign(texto="Produto: X | Nota: 5/5 | Avaliação: ligue 81 98888-7777")
print("\nteste do gate com vazamento:", gates_chunks(vazado).query("status == 'ERRO'")["regra"].tolist())

In [ ]:
if (g["status"] == "ERRO").any():
    raise RuntimeError("Gates dos chunks falharam: NÃO publicar.")

cols = ["chunk_id", "fonte", "fonte_id", "texto", "metadata", "content_hash", "politica_pii",
        "embedding_status", "modelo_embedding", "embedding_em", "atualizado_em"]
gravar_gold(chunks[cols], "chunks")
print(chunks["embedding_status"].value_counts().to_string())
print("removidos desde a última publicação:", len(removidos))
print("publicado em s3://warehouse/gold/chunks/")

### Passo 11 · O contrato com o próximo módulo
Esta é a tabela que o banco vetorial vai ter. **Ela é apenas escrita aqui, não é criada.** Repare que `embedding` e o índice ficam em aberto: a **dimensão** depende do modelo de embedding e o **índice** depende do volume; ambos são decisões do próximo módulo.

In [ ]:
DDL_PGVECTOR = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS gold.chunks (
    chunk_id          text PRIMARY KEY,         -- 'avaliacao:123' (estável entre execuções)
    fonte             text        NOT NULL,     -- de qual tabela veio
    fonte_id          text        NOT NULL,
    texto             text        NOT NULL,     -- já com PII mascarada
    metadata          jsonb       NOT NULL,     -- filtros de busca (produto, nota); sem dado pessoal
    content_hash      char(64)    NOT NULL,     -- sha256 do texto normalizado
    politica_pii      text        NOT NULL,     -- versão da regra de mascaramento aplicada
    embedding         vector(<DIM>),            -- DIM: definido no módulo vetorial (depende do modelo)
    modelo_embedding  text,                     -- qual modelo gerou o vetor
    embedding_em      timestamptz,
    embedding_status  text        NOT NULL DEFAULT 'pendente',
    atualizado_em     timestamptz NOT NULL
);
-- índice (HNSW ou IVFFlat) e filtros por metadata: decididos no módulo vetorial
"""

CONTRATO_VETORIAL = {
    "tabela_origem": "s3://warehouse/gold/chunks/",
    "grao": "um chunk",
    "chave": "chunk_id",
    "dono": "Dados / IA (exemplo: preencha com o responsável)",
    "garantias": ["texto com PII mascarada pela política " + VERSAO_POLITICA_PII,
                  "texto com no máximo 1000 caracteres", "content_hash = sha256 do texto normalizado",
                  "metadados sem cliente_id, nome ou e-mail"],
    "regra_incremental": "gerar embedding apenas quando embedding_status = 'pendente'",
    "fonte_da_verdade": "o lake (Parquet); o banco vetorial é derivado e pode ser recriado",
    "pendente_no_proximo_modulo": ["modelo e dimensão do embedding", "índice", "estratégia de chunking para textos longos",
                                   "NER para nomes próprios em texto livre"],
    "ddl_referencia": DDL_PGVECTOR.strip(),
}
s3.put_object(Bucket="warehouse", Key="gold/_contratos/contrato_vetorial.json",
              Body=json.dumps(CONTRATO_VETORIAL, ensure_ascii=False, indent=2).encode("utf-8"))
print(DDL_PGVECTOR)
print("contrato salvo em s3://warehouse/gold/_contratos/contrato_vetorial.json")

### Passo 12 · Status: o que o próximo módulo recebe

In [ ]:
pub = ler_prefixo("warehouse", "gold/chunks/")
print("chunks publicados :", len(pub))
print("estado dos embeddings:")
print(pub["embedding_status"].value_counts().to_string())
print("\nobjetos na Gold:")
for chave, tam in listar("warehouse", "gold/"):
    print(f"{tam:>9,}  {chave}")
pub[["chunk_id", "texto", "embedding_status"]].head(3)

## Qualidade e governança neste notebook
- **Contrato antes do código:** grão, dono e descrição de cada coluna. O gate recusa coluna sem descrição.
- **Reconciliação:** a receita da Gold foi recalculada por outro caminho e comparada com a Silver.
- **Minimização:** e-mail fora da Gold, nome mascarado, `cliente_id` fora dos chunks.
- **Menor privilégio:** o Metabase lê com um usuário que só tem `SELECT`.
- **Fonte da verdade:** o lake guarda a Gold; o Postgres e o banco vetorial são cópias derivadas.
- **Versão da política de PII:** cada chunk registra qual regra o mascarou, então dá para reprocessar quando a regra mudar.

## Checkpoint
- [x] `gold/vendas_diarias`, `gold/clientes_360`, `gold/avaliacoes_resumo` e `gold/chunks` existem no bucket `warehouse`.
- [x] Você explica por que a publicação é bloqueada quando um gate falha.
- [x] Você explica por que a PII é mascarada **antes** de gerar embeddings.
- [x] Você explica o que o `content_hash` economiza no próximo módulo.
- [x] Você sabe dizer o que ficou **de fora** de propósito (o banco vetorial) e por quê.

➡️ Próximo módulo: embeddings e banco vetorial (pgvector)